# Polaris–Iceberg Data Lifecycle Practice



> **New here? Read this first.** This notebook follows a single dataset through its
> entire life so you can SEE how Iceberg versions data — written, appended,
> time-traveled, rolled back, evolved, orphaned, and reclaimed.

**Focus: the DATA.** Catalog/namespace/table are just scaffolding to hold it.

```
Part 0   Setup (brief scaffolding)
Part 1   Write data → snapshot #1
Part 2   Read the data back (full rows)
Part 3   Append data → snapshot #2 (TRUE cumulative append — history grows)
Part 4   Time travel — read actual rows as-of #1 (5 rows) vs current (10 rows)
Part 5   Rollback to #1, then roll forward to #2
Part 6   Schema evolution — add a column (new metadata version)
Part 7   Orphan detection — live file set vs MinIO file set (manifest walk)
Part 8   Drop & purge — what happens to the DATA (verified, no STS)
Part 9   Reclaim storage — delete orphans with static MinIO creds
Part 10  Summary — the data's journey
```

## Key concepts you'll see in action

```
snapshot         a complete, queryable version of the table (enables time travel)
manifest chain   snapshot → manifest-list → manifests → data files
cumulative append a new snapshot carries forward the parent's manifests, so the
                 table ACCUMULATES rows (5 → 10) rather than replacing them
orphan           a file in MinIO that no live snapshot references
```

## Prerequisites

```
Files (same folder):  polaris_test_utils.py, minio_rest.py, config/
Packages:             pip install requests pyyaml opensearch-py pyarrow fastavro
Access:               Polaris + MinIO reachable (URLs shown by the env banner)
```

## How to run

```
1. Run the setup cell — it prints 🟢 DEV / 🔴 PROD so you know your target.
2. Run top to bottom. Each part prints what it proves.
3. Switch env: change init_env("dev") → init_env("prod") in setup, re-run it.

⚠️  All storage I/O is via the MinIO REST API (minio_rest.py / `mc`) — NO s3fs
    (company policy). Manifests are encoded with fastavro (not a storage client).
    No credentials are hardcoded; they come from config or environment variables.

NOTE on cumulative append (Part 3): the new snapshot's manifest-list carries
forward the parent's manifests, re-labeling them added→existing. This depends on
Polaris accepting the carried entries; if Part 3's commit is rejected, check the
error — manifest-list validation strictness can vary by Polaris version.
```

---


## Part 0 — Setup (scaffolding to hold the data)

In [56]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_ROOT = pathlib.Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import io, os, time, uuid
import json as _json
import pyarrow as pa

try:
    import fastavro
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "fastavro", "--quiet", "--break-system-packages"])
    import fastavro

from iceberg_rest import IcebergREST
from minio_rest import MinioREST
from polaris_rest import PolarisREST

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — inline, no init_env() / config yaml.
#
# Endpoints and names are hardcoded so this notebook runs standalone. Secrets
# read from the environment with a visible local default, so it still runs with
# zero setup but nothing production-shaped is baked into a tracked file:
#
#     export POLARIS_ROOT_SECRET=...   MINIO_ACCESS_KEY=...   MINIO_SECRET_KEY=...
#
# CLAUDE.md forbids plaintext credentials in notebooks. The defaults below are
# the throwaway values of a disposable LOCAL cluster — do not point this file at
# a shared environment (the guard below refuses to).
# ══════════════════════════════════════════════════════════════════════════════
POLARIS_URL    = "http://192.168.139.2:8181"
REALM          = "POLARIS"
ROOT_CLIENT    = "root"
ROOT_SECRET    = os.environ.get("POLARIS_ROOT_SECRET", "polaris-secret")

MINIO_ENDPOINT = "http://192.168.139.2:9000"
MINIO_ACCESS   = os.environ.get("MINIO_ACCESS_KEY", "admin")
MINIO_SECRET   = os.environ.get("MINIO_SECRET_KEY", "f4ua4XYGIInUe8GtWljrOS09vKAVrINP")
BUCKET         = "data-catalog-bucket"

# Replaces require_not_prod(): this notebook drops catalogs and purges storage,
# so it must stay incapable of doing that to a shared cluster.
assert any(hp in POLARIS_URL for hp in ("localhost", "127.0.0.1", "192.168.")), (
    f"Refusing to run against {POLARIS_URL} — this notebook is LOCAL-ONLY "
    "(it drops catalogs and deletes objects)."
)

# ── clients ───────────────────────────────────────────────────────────────────
# pc  — Management API as ROOT: catalogs, principals, roles, grants.
# ic  — Iceberg API as ROOT: used only for bootstrap/cleanup.
# icd — Iceberg API as the dedicated INSTANCE PRINCIPAL, created in Part 0.
#       Every data operation runs through icd, not root, so the notebook
#       exercises the same privilege path a real engine would.
pc = PolarisREST(POLARIS_URL, REALM)
pc.token = pc.get_token(ROOT_CLIENT, ROOT_SECRET).json()["access_token"]
ic = IcebergREST(POLARIS_URL, REALM, token=pc.token)
icd = ic  # rebound in Part 0 once the instance principal exists

mc = MinioREST(MINIO_ENDPOINT, MINIO_ACCESS, MINIO_SECRET, BUCKET)

# A cluster rebuild takes the bucket with it. Without this, the failure surfaces
# much later as a Polaris create-table error — "The specified bucket does not
# exist (Service: S3, Status Code: 404)" — which reads like a catalog problem.
if mc.ensure_bucket():
    print(f"   created missing bucket: {BUCKET}")


def show(resp, label=""):
    """Compact request result line (replaces nb_support.print_response)."""
    ok = "✅" if resp.status_code < 300 else "❌"
    body = "" if resp.status_code in (204,) else f" {resp.text[:160]}"
    print(f"{ok} {label} [{resp.status_code}]{body}")
    return resp


# Data lifecycle resource names
CAT = "data-lifecycle-practice"
NS = "life-ns"
TBL = "events"
LOC = f"s3a://{BUCKET}/{CAT}/"
TBL_LOC = f"{LOC}{NS}/{TBL}"

print(f"✅ clients ready — Polaris {POLARIS_URL} realm={REALM}, MinIO bucket={BUCKET}")

✅ clients ready — Polaris http://192.168.139.2:8181 realm=POLARIS, MinIO bucket=data-catalog-bucket


In [57]:
# Idempotent cleanup of any leftover from prior runs
print("Cleanup leftover resources...")

catalogs = [c["name"] for c in pc.list_catalogs().json().get("catalogs", [])]
if CAT in catalogs:
    # Enable purge + grant so we can fully clear. update_catalog REPLACES the
    # property map, so merge into the existing one or default-base-location
    # (and the storage config with it) is lost.
    cat = pc.get_catalog(CAT).json()
    pc.update_catalog(
        CAT,
        {
            **cat.get("properties", {}),
            "polaris.config.drop-with-purge.enabled": "true",
            "polaris.config.purge-view-metadata-on-drop": "false",
        },
        cat["entityVersion"],
    )
    pc.grant_privilege(CAT, "catalog_admin", "CATALOG_MANAGE_CONTENT")

    for ns in ic.list_namespaces(CAT).json().get("namespaces", []):
        nsn = ns[0] if isinstance(ns, list) else ns
        for t in ic.list_tables(CAT, nsn).json().get("identifiers", []):
            ic.drop_table(CAT, nsn, t["name"], purge=True)
        ic.drop_namespace(CAT, nsn)

    pc.delete_catalog(CAT, purge=True)

# Sweep storage (one object at a time — bulk delete needs Content-MD5 on MinIO)
mc.delete_prefix(f"{CAT}/", min_size=0)
print("✅ clean")

Cleanup leftover resources...
✅ clean


In [58]:
# Create scaffolding: catalog + namespace + table (schema: id, name, amount)
print("Create catalog + namespace + table")

show(
    pc.create_catalog(
        CAT,
        bucket=BUCKET,
        minio_endpoint=MINIO_ENDPOINT,
        properties={
            "default-base-location": LOC,
            "polaris.config.drop-with-purge.enabled": "false",
            "polaris.config.purge-view-metadata-on-drop": "false",
        },
    ),
    "Create catalog",
)
pc.grant_privilege(CAT, "catalog_admin", "CATALOG_MANAGE_CONTENT")


def instance_principal(catalog, suffix=None):
    """Provision a dedicated principal + roles + grant for DATA operations.

    Replaces nb_support.instance_principal, rebuilt on PolarisREST.
    Order matters: the catalog-role must exist and be visible before it can be
    granted to, which is why the assignments come before the grant.

    Returns:
        (IcebergREST bound to the new principal, (principal, prole, crole))
    """
    sfx = suffix or uuid.uuid4().hex[:8]
    p, pr, cr = f"{catalog}-{sfx}-principal", f"{catalog}-{sfx}-prole", f"{catalog}-{sfx}-crole"

    body = pc.create_principal(p).json()
    client_id = body["principal"]["clientId"]
    client_secret = body["credentials"]["clientSecret"]

    pc.create_principal_role(pr)
    pc.create_catalog_role(catalog, cr)
    pc.assign_principal_role_to_principal(p, pr)
    pc.assign_catalog_role_to_principal_role(catalog, pr, cr)
    pc.grant_privilege(catalog, cr, "CATALOG_MANAGE_CONTENT")

    tok = pc.get_token(client_id, client_secret).json()["access_token"]
    return IcebergREST(POLARIS_URL, REALM, token=tok), (p, pr, cr)


# From here on, DATA operations run as a dedicated INSTANCE PRINCIPAL, not root.
# Root bootstrapped the catalog + grants above; the instance principal owns the
# namespace/table/snapshot/read/drop lifecycle below.
icd, _inst_names = instance_principal(CAT)
print(f"   instance principal: {_inst_names[0]}")

show(icd.create_namespace(CAT, NS), "Create namespace")

show(
    icd.create_table(
        CAT,
        NS,
        {
            "name": TBL,
            "location": f"{TBL_LOC}/",
            "schema": {
                "type": "struct",
                "schema-id": 0,
                "fields": [
                    {"id": 1, "name": "id", "type": "long", "required": True},
                    {"id": 2, "name": "name", "type": "string", "required": False},
                    {"id": 3, "name": "amount", "type": "double", "required": False},
                ],
            },
            "properties": {"write.format.default": "parquet"},
        },
    ),
    "Create table",
)
print("\n✅ scaffolding ready — now the data lifecycle begins")

Create catalog + namespace + table
❌ Create catalog [409] {"error":{"message":"Cannot create Catalog data-lifecycle-practice. Catalog already exists or resolution failed","type":"AlreadyExistsException","code":409}}
   instance principal: data-lifecycle-practice-962b6e3f-principal
✅ Create namespace [200] {"namespace":["life-ns"],"properties":{"location":"s3a://data-catalog-bucket/data-lifecycle-practice/life-ns/"}}
✅ Create table [200] {"metadata-location":"s3a://data-catalog-bucket/data-lifecycle-practice/life-ns/events/metadata/00000-407a5112-9bdb-4e11-a49e-3f724bf990fe.metadata.json","metad

✅ scaffolding ready — now the data lifecycle begins


### Shared helpers — write a snapshot, read data, walk manifests

These functions are the core of the notebook. `write_snapshot()` writes a Parquet
file + Iceberg manifest/manifest-list and commits a snapshot. `read_table_data()`
reads the actual rows. `live_data_files()` walks snapshot→manifest-list→manifests
to find which data files the table currently references (used for orphan detection).


In [59]:
# ---- Iceberg Avro schemas (v2) ----
MANIFEST_SCHEMA = {
    "type":"record","name":"manifest_entry","fields":[
        {"name":"status","type":"int"},
        {"name":"snapshot_id","type":["null","long"],"default":None},
        {"name":"sequence_number","type":["null","long"],"default":None},
        {"name":"file_sequence_number","type":["null","long"],"default":None},
        {"name":"data_file","type":{"type":"record","name":"r2","fields":[
            {"name":"content","type":"int"},
            {"name":"file_path","type":"string"},
            {"name":"file_format","type":"string"},
            {"name":"partition","type":{"type":"record","name":"r102","fields":[]}},
            {"name":"record_count","type":"long"},
            {"name":"file_size_in_bytes","type":"long"},
            {"name":"column_sizes","type":["null",{"type":"array","items":{"type":"record","name":"k81v81","fields":[{"name":"key","type":"int"},{"name":"value","type":"long"}]}}],"default":None},
            {"name":"value_counts","type":["null",{"type":"array","items":"k81v81"}],"default":None},
            {"name":"null_value_counts","type":["null",{"type":"array","items":"k81v81"}],"default":None},
            {"name":"nan_value_counts","type":["null",{"type":"array","items":"k81v81"}],"default":None},
            {"name":"lower_bounds","type":["null",{"type":"array","items":{"type":"record","name":"k81v100","fields":[{"name":"key","type":"int"},{"name":"value","type":"bytes"}]}}],"default":None},
            {"name":"upper_bounds","type":["null",{"type":"array","items":"k81v100"}],"default":None},
            {"name":"key_metadata","type":["null","bytes"],"default":None},
            {"name":"split_offsets","type":["null",{"type":"array","items":"long"}],"default":None},
            {"name":"equality_ids","type":["null",{"type":"array","items":"int"}],"default":None},
            {"name":"sort_order_id","type":["null","int"],"default":None},
        ]}}
    ]
}
MANIFEST_LIST_SCHEMA = {
    "type":"record","name":"manifest_file","fields":[
        {"name":"manifest_path","type":"string"},
        {"name":"manifest_length","type":"long"},
        {"name":"partition_spec_id","type":"int"},
        {"name":"content","type":"int","default":0},
        {"name":"sequence_number","type":"long","default":0},
        {"name":"min_sequence_number","type":"long","default":0},
        {"name":"added_snapshot_id","type":"long"},
        {"name":"added_data_files_count","type":"int"},
        {"name":"existing_data_files_count","type":"int"},
        {"name":"deleted_data_files_count","type":"int"},
        {"name":"added_rows_count","type":"long"},
        {"name":"existing_rows_count","type":"long"},
        {"name":"deleted_rows_count","type":"long"},
        {"name":"partitions","type":["null",{"type":"array","items":{"type":"record","name":"r508","fields":[
            {"name":"contains_null","type":"boolean"},
            {"name":"contains_nan","type":["null","boolean"],"default":None},
            {"name":"lower_bound","type":["null","bytes"],"default":None},
            {"name":"upper_bound","type":["null","bytes"],"default":None}]}}],"default":None},
        {"name":"key_metadata","type":["null","bytes"],"default":None},
    ]
}
print("✅ Iceberg Avro schemas defined")


✅ Iceberg Avro schemas defined


In [60]:
def _read_manifest_list(ml_path):
    """Return list of manifest_file records from an existing manifest-list."""
    return list(fastavro.reader(io.BytesIO(mc.get_bytes(ml_path))))

def write_snapshot(rows, schema_json, label="", parent_snapshot_id=None):
    """Write batch as a new manifest, then build a manifest-list that includes
    BOTH the new manifest AND all manifests carried forward from the parent
    snapshot (true cumulative append). Returns snapshot_id.

    parent_snapshot_id: if given, this snapshot accumulates the parent's data.
    """
    snap_id = int(time.time() * 1000)
    # Iceberg v2: each snapshot's sequence-number MUST be strictly greater than
    # the table's last-sequence-number. Compute it as last+1 for EVERY snapshot.
    meta = icd.load_table(CAT, NS, TBL).json()["metadata"]
    snaps = {s["snapshot-id"]: s for s in meta.get("snapshots", [])}
    last_seq = meta.get("last-sequence-number", 0) or 0
    seq = last_seq + 1

    nrows = len(rows)
    data_dir = f"{TBL_LOC}/data"
    meta_dir = f"{TBL_LOC}/metadata"

    # 1. Parquet for THIS batch
    pq_path = f"{data_dir}/{uuid.uuid4().hex}.parquet"
    mc.put_parquet_table(pq_path, rows)
    fsize = mc.size(pq_path)

    # 2. manifest for THIS batch (status=1 ADDED)
    entry = {"status":1,"snapshot_id":snap_id,"sequence_number":seq,"file_sequence_number":seq,
        "data_file":{"content":0,"file_path":pq_path,"file_format":"PARQUET","partition":{},
            "record_count":nrows,"file_size_in_bytes":fsize,"column_sizes":None,"value_counts":None,
            "null_value_counts":None,"nan_value_counts":None,"lower_bounds":None,"upper_bounds":None,
            "key_metadata":None,"split_offsets":None,"equality_ids":None,"sort_order_id":0}}
    man_path = f"{meta_dir}/{uuid.uuid4().hex}-m{seq}.avro"
    buf = io.BytesIO()
    fastavro.writer(buf, fastavro.parse_schema(MANIFEST_SCHEMA), [entry], metadata={
        "schema": schema_json, "partition-spec":"[]","partition-spec-id":"0","format-version":"2",
        "content":"data","sequence-number":str(seq),"min-sequence-number":str(seq),
        "snapshot-id":str(snap_id),"added-data-files":"1","existing-data-files":"0",
        "deleted-data-files":"0","added-rows":str(nrows),"existing-rows":"0","deleted-rows":"0"})
    man_bytes = buf.getvalue()
    mc.put_bytes(man_path, man_bytes)

    # 3. manifest-list = new manifest + ALL manifests carried forward from parent
    ml_entries = [{
        "manifest_path":man_path,"manifest_length":len(man_bytes),"partition_spec_id":0,
        "content":0,"sequence_number":seq,"min_sequence_number":seq,"added_snapshot_id":snap_id,
        "added_data_files_count":1,"existing_data_files_count":0,"deleted_data_files_count":0,
        "added_rows_count":nrows,"existing_rows_count":0,"deleted_rows_count":0,
        "partitions":None,"key_metadata":None}]

    carried_rows = 0
    if parent_snapshot_id and parent_snapshot_id in snaps:
        parent_ml = snaps[parent_snapshot_id]["manifest-list"]
        for prev in _read_manifest_list(parent_ml):
            # carry forward as EXISTING data (re-label added→existing for clarity)
            carried = dict(prev)
            carried["existing_data_files_count"] = (prev.get("added_data_files_count",0)
                                                    + prev.get("existing_data_files_count",0))
            carried["added_data_files_count"] = 0
            carried["existing_rows_count"] = (prev.get("added_rows_count",0)
                                              + prev.get("existing_rows_count",0))
            carried_rows += carried["existing_rows_count"]
            carried["added_rows_count"] = 0
            ml_entries.append(carried)

    ml_path = f"{meta_dir}/snap-{snap_id}-{uuid.uuid4().hex}.avro"
    buf2 = io.BytesIO()
    fastavro.writer(buf2, fastavro.parse_schema(MANIFEST_LIST_SCHEMA), ml_entries)
    mc.put_bytes(ml_path, buf2.getvalue())

    total_records = nrows + carried_rows
    total_files = len(ml_entries)

    # 4. commit add-snapshot (with parent-snapshot-id + cumulative totals)
    snap_body = {
        "snapshot-id":snap_id,"timestamp-ms":snap_id,
        "sequence-number":seq,                       # ← REQUIRED: must be > last-sequence-number
        "summary":{"operation":"append","added-data-files":"1","added-records":str(nrows),
                   "added-files-size":str(fsize),"total-records":str(total_records),
                   "total-data-files":str(total_files)},
        "schema-id":0,"manifest-list":ml_path}
    if parent_snapshot_id:
        snap_body["parent-snapshot-id"] = parent_snapshot_id
    r_add = icd.commit_table(CAT, NS, TBL,
        [{"action": "add-snapshot", "snapshot": snap_body}])
    if r_add.status_code not in (200, 204):
        print(f"  ❌ add-snapshot FAILED [{r_add.status_code}]: {r_add.text[:300]}")
        raise RuntimeError(f"add-snapshot rejected: {r_add.status_code}")
    # 5. set main ref
    r_ref = icd.commit_table(CAT, NS, TBL,
        [{"action": "set-snapshot-ref", "ref-name": "main",
          "type": "branch", "snapshot-id": snap_id}])
    if r_ref.status_code not in (200, 204):
        print(f"  ❌ set-snapshot-ref FAILED [{r_ref.status_code}]: {r_ref.text[:300]}")
        raise RuntimeError(f"set-snapshot-ref rejected: {r_ref.status_code}")
    # verify it took
    chk = icd.load_table(CAT, NS, TBL).json()["metadata"]
    if chk.get("current-snapshot-id") != snap_id:
        print(f"  ⚠️ commit returned OK but current-snapshot-id={chk.get('current-snapshot-id')} (expected {snap_id})")
    print(f"  ✅ snapshot {snap_id} committed (+{nrows} rows, total {total_records}) {label}")
    return snap_id

print("✅ write_snapshot() defined — TRUE cumulative append (carries forward parent manifests)")

✅ write_snapshot() defined — TRUE cumulative append (carries forward parent manifests)


In [61]:
def read_table_data(snapshot_id=None):
    """Read the ACTUAL rows the table references. If snapshot_id given, read that
    snapshot's data (time travel); else read current. Walks manifest-list →
    manifest → data files, then loads the Parquet."""
    meta = icd.load_table(CAT, NS, TBL).json()["metadata"]
    snaps = {s["snapshot-id"]: s for s in meta.get("snapshots", [])}
    target = snapshot_id if snapshot_id is not None else meta.get("current-snapshot-id")
    if target is None or target == -1:
        print(f"  ⚠️ table has no current snapshot (current-snapshot-id={target}). "
              f"The write/commit likely did not take — check Part 1 output.")
        return None
    if target not in snaps:
        print(f"  ⚠️ snapshot {target} not in table history {list(snaps.keys())}")
        return None
    ml_path = snaps[target]["manifest-list"]

    # read manifest-list → manifest paths
    manifests = [r["manifest_path"] for r in fastavro.reader(io.BytesIO(mc.get_bytes(ml_path)))]
    # read each manifest → data file paths (status != 2 = not deleted)
    data_files = []
    for mp in manifests:
        for r in fastavro.reader(io.BytesIO(mc.get_bytes(mp))):
            if r["status"] != 2:
                data_files.append(r["data_file"]["file_path"])
    # load + concat the Parquet files
    tables = []
    for dp in data_files:
        tables.append(mc.get_parquet_table(dp))
    if not tables:
        print(f"  ⚠️ snapshot {target} references no data files "
              f"(manifests={len(manifests)}, data_files={len(data_files)})")
        return None
    combined = pa.concat_tables(tables) if len(tables) > 1 else tables[0]
    return combined.to_pandas()

def live_data_files(snapshot_id=None):
    """Return the SET of data+metadata files the table references at a snapshot
    (current if None). Used for orphan detection: live set vs MinIO set."""
    meta = icd.load_table(CAT, NS, TBL).json()["metadata"]
    snaps = {s["snapshot-id"]: s for s in meta.get("snapshots", [])}
    target = snapshot_id if snapshot_id is not None else meta.get("current-snapshot-id")
    live = set()
    if target in snaps:
        ml = snaps[target]["manifest-list"]
        live.add(mc._key(ml))
        for r in fastavro.reader(io.BytesIO(mc.get_bytes(ml))):
            mp = r["manifest_path"]; live.add(mc._key(mp))
            for e in fastavro.reader(io.BytesIO(mc.get_bytes(mp))):
                if e["status"] != 2:
                    live.add(mc._key(e["data_file"]["file_path"]))
    return live

print("✅ read_table_data() and live_data_files() defined")

✅ read_table_data() and live_data_files() defined


## Part 1 — Write data → snapshot #1

In [62]:
SCHEMA_V1 = '{"type":"struct","schema-id":0,"fields":[{"id":1,"name":"id","required":true,"type":"long"},{"id":2,"name":"name","required":false,"type":"string"},{"id":3,"name":"amount","required":false,"type":"double"}]}'

batch1 = pa.table({
    "id":     pa.array([1,2,3,4,5], type=pa.int64()),
    "name":   pa.array(["alice","bob","carol","dave","eve"], type=pa.string()),
    "amount": pa.array([100.0,200.5,300.0,150.75,250.0], type=pa.float64()),
})
print("Writing snapshot #1 (5 rows)")
snap1 = write_snapshot(batch1, SCHEMA_V1, "← initial load")  # first snapshot, no parent


Writing snapshot #1 (5 rows)
  ✅ snapshot 1787126884475 committed (+5 rows, total 5) ← initial load


## Part 2 — Read the data back (full rows)

In [55]:
print("Reading current table data:\n")
df = read_table_data()
print(df.to_string(index=False))
print(f"\n✅ {len(df)} rows read back from snapshot #1")


Reading current table data:

 id  name  amount
  1 alice  100.00
  2   bob  200.50
  3 carol  300.00
  4  dave  150.75
  5   eve  250.00

✅ 5 rows read back from snapshot #1


## Part 3 — Append data → snapshot #2 (history grows)

In [42]:
# TRUE cumulative append: snapshot #2 carries forward snapshot #1's
# manifest, so it references BOTH batches (10 rows total). Time-travel
# to #1 still shows only the original 5 rows.
batch2 = pa.table({
    "id":     pa.array([6,7,8,9,10], type=pa.int64()),
    "name":   pa.array(["frank","grace","heidi","ivan","judy"], type=pa.string()),
    "amount": pa.array([400.0,500.0,600.0,700.0,800.0], type=pa.float64()),
})
print("Writing snapshot #2 (5 more rows)")
snap2 = write_snapshot(batch2, SCHEMA_V1, "← appended", parent_snapshot_id=snap1)

meta = icd.load_table(CAT, NS, TBL).json()["metadata"]
print(f"\nSnapshots now in table history: {len(meta.get('snapshots', []))}")
for s in meta.get("snapshots", []):
    print(f"  snapshot {s['snapshot-id']}  op={s['summary'].get('operation')}  records={s['summary'].get('total-records','?')}")
print(f"current-snapshot-id: {meta.get('current-snapshot-id')}")

Writing snapshot #2 (5 more rows)
  ✅ snapshot 1787125367328 committed (+5 rows, total 10) ← appended

Snapshots now in table history: 2
  snapshot 1787125365035  op=append  records=5
  snapshot 1787125367328  op=append  records=10
current-snapshot-id: 1787125367328


## Part 4 — Time travel: read actual rows as-of each snapshot

**This is Iceberg time travel.** Each snapshot is a complete, queryable version of
the table. We read the real rows at snapshot #1 vs the current snapshot.


In [43]:
print("=== Time travel ===\n")
print(f"AS OF snapshot #1 ({snap1}) — original load only:")
df1 = read_table_data(snap1)
print(df1.to_string(index=False))
print(f"  → {len(df1)} rows\n")

print(f"AS OF snapshot #2 ({snap2}) — current, cumulative:")
df2 = read_table_data(snap2)
print(df2.to_string(index=False))
print(f"  → {len(df2)} rows\n")

print("✅ Time travel: snapshot #1 has the original 5 rows; snapshot #2 has all 10")
print("   (batch1 carried forward + batch2 appended). Same table, two versions.")


=== Time travel ===

AS OF snapshot #1 (1787125365035) — original load only:
 id  name  amount
  1 alice  100.00
  2   bob  200.50
  3 carol  300.00
  4  dave  150.75
  5   eve  250.00
  → 5 rows

AS OF snapshot #2 (1787125367328) — current, cumulative:
 id  name  amount
  6 frank  400.00
  7 grace  500.00
  8 heidi  600.00
  9  ivan  700.00
 10  judy  800.00
  1 alice  100.00
  2   bob  200.50
  3 carol  300.00
  4  dave  150.75
  5   eve  250.00
  → 10 rows

✅ Time travel: snapshot #1 has the original 5 rows; snapshot #2 has all 10
   (batch1 carried forward + batch2 appended). Same table, two versions.


## Part 5 — Rollback to #1, then roll forward to #2

Rollback = move the `main` branch ref back to an earlier snapshot. The table's
"current" view changes, but no data is deleted — snapshot #2's files still exist
(this creates a natural orphan, used in Part 7).


In [44]:
def set_main_to(snapshot_id):
    return icd.commit_table(CAT, NS, TBL,
        [{"action": "set-snapshot-ref", "ref-name": "main",
          "type": "branch", "snapshot-id": snapshot_id}])

def current_snap():
    return icd.load_table(CAT, NS, TBL).json()["metadata"]["current-snapshot-id"]

print("ROLLBACK main → snapshot #1")
set_main_to(snap1)
print(f"  current-snapshot-id now: {current_snap()}  (== snap1: {current_snap() == snap1})")
df_rb = read_table_data()
print(f"  current data ({len(df_rb)} rows):")
print(df_rb.to_string(index=False))
print()

print("ROLL FORWARD main → snapshot #2")
set_main_to(snap2)
print(f"  current-snapshot-id now: {current_snap()}  (== snap2: {current_snap() == snap2})")
print(f"  current row count: {len(read_table_data())}")
print()
print("✅ Rollback/forward just moves the ref. No data rewritten or deleted.")

ROLLBACK main → snapshot #1
  current-snapshot-id now: 1787125365035  (== snap1: True)
  current data (5 rows):
 id  name  amount
  1 alice  100.00
  2   bob  200.50
  3 carol  300.00
  4  dave  150.75
  5   eve  250.00

ROLL FORWARD main → snapshot #2
  current-snapshot-id now: 1787125367328  (== snap2: True)
  current row count: 10

✅ Rollback/forward just moves the ref. No data rewritten or deleted.


## Part 6 — Schema evolution: add a column

Iceberg evolves schema in place — adding a column is a metadata change, no data
rewrite. Existing rows read back with `null` for the new column. This is a core
part of a data lifecycle: the shape changes over time without breaking history.


In [45]:
print("Add column 'category' (string) via update-schema")
# Read current schema, bump schema-id, add field id 4
meta = icd.load_table(CAT, NS, TBL).json()["metadata"]
cur_schema = [s for s in meta["schemas"] if s["schema-id"] == meta["current-schema-id"]][0]
new_fields = cur_schema["fields"] + [
    {"id": 4, "name": "category", "required": False, "type": "string"}
]

r = icd.commit_table(CAT, NS, TBL, [
    {"action": "add-schema",
     "schema": {"type": "struct", "schema-id": 1, "fields": new_fields}},
    {"action": "set-current-schema", "schema-id": 1},
])
print(f"  update-schema: HTTP {r.status_code}")

meta2 = icd.load_table(CAT, NS, TBL).json()["metadata"]
print(f"  current-schema-id: {meta2.get('current-schema-id')}")
print(f"  fields now: {[f['name'] for f in [s for s in meta2['schemas'] if s['schema-id']==meta2['current-schema-id']][0]['fields']]}")
print()
print("✅ Column added as a metadata change — existing data untouched.")
print("   New writes can populate 'category'; old rows read back null for it.")

Add column 'category' (string) via update-schema
  update-schema: HTTP 200
  current-schema-id: 1
  fields now: ['id', 'name', 'amount', 'category']

✅ Column added as a metadata change — existing data untouched.
   New writes can populate 'category'; old rows read back null for it.


## Part 7 — Orphan detection: live file set vs MinIO file set

**Polaris API has no orphan-file endpoint.** An orphan is a file in MinIO that no
live snapshot references. To find them you compare:
```
  (A) files that EXIST in MinIO          ← ask MinIO
  (B) files the LIVE table REFERENCES    ← walk snapshot→manifest-list→manifests
  orphans = A − B
```
With cumulative appends, snapshot #2 references BOTH data files (batch1 + batch2),
so both `.parquet` files are live. But snapshot #1's **old manifest-list** is now
superseded — snapshot #2 has its own manifest-list. Those superseded metadata files
are orphans. This walks the live set and shows them.


In [46]:
# (A) everything in MinIO under the table (ignore 0-byte directory markers)
actual = set(mc.list(f"{CAT}/{NS}/{TBL}/"))   # recursive, >0 bytes, no cache

# (B) files the CURRENT snapshot references (walks manifest chain)
live = live_data_files()  # current snapshot

orphans = actual - live
print(f"Files in MinIO (real):           {len(actual)}")
print(f"Files referenced by current snap: {len(live)}")
print(f"Orphans (in MinIO, unreferenced): {len(orphans)}\n")

# categorize
for o in sorted(orphans):
    kind = "data" if "/data/" in o else "metadata"
    print(f"  [{kind}] {o.split('/')[-1]}")
print()
print("With cumulative append, both .parquet data files are LIVE (snapshot #2")
print("references batch1 + batch2). The orphans are superseded manifest-list /")
print("manifest files from snapshot #1 and the rollback experiment.")
print()
print("Polaris won't report these — only this Iceberg-reference walk finds them.")


Files in MinIO (real):           14
Files referenced by current snap: 5
Orphans (in MinIO, unreferenced): 9

  [metadata] 00000-feac364c-7942-4bce-901c-6ae476be97ae.metadata.json
  [metadata] 00001-03e4d034-0679-4efc-be1c-52bca95083bb.metadata.json
  [metadata] 00002-cbb3e00d-8ef1-4038-ac87-e4515050a3a2.metadata.json
  [metadata] 00003-510c5b41-24be-4d9a-8a03-dc930c5ce850.metadata.json
  [metadata] 00004-f9b65c53-783e-41a4-a4f3-0c8b364e923c.metadata.json
  [metadata] 00005-cb4f3f33-a967-427a-ad3e-780720f37e72.metadata.json
  [metadata] 00006-02ae1fec-5d48-47af-9de6-10f27eb005ee.metadata.json
  [metadata] 00007-ea3d623e-b471-47c7-8aa6-9d688f309645.metadata.json
  [metadata] snap-1787125365035-0d19c850b21e497a9321609c586bb2b3.avro

With cumulative append, both .parquet data files are LIVE (snapshot #2
references batch1 + batch2). The orphans are superseded manifest-list /
manifest files from snapshot #1 and the rollback experiment.

Polaris won't report these — only this Iceberg-referenc

## Part 8 — Drop & purge: what happens to the DATA (verified)

When the table is dropped, its catalog record is removed. With `purgeRequested=true`,
Polaris *schedules* a cleanup task to delete the data files — **but in this no-STS
environment that task fails and the files orphan** (see the purge troubleshooting
doc / issue #379). We verify the data files remain after purge.


In [47]:
# Enable purge + ensure grant, then drop the table WITH purge
cat = pc.get_catalog(CAT).json()
pc.update_catalog(
    CAT,
    {**cat.get("properties", {}), "polaris.config.drop-with-purge.enabled": "true"},
    cat["entityVersion"],
)

before = mc.list(f"{CAT}/{NS}/{TBL}/")
print(f"Data/metadata files before purge: {len(before)}")

# NOTE: table purge is gated by a DISTINCT op, DROP_TABLE_WITH_PURGE, which root
# does NOT bypass — TABLE_DROP alone is insufficient and CATALOG_MANAGE_CONTENT
# is required. The instance principal was granted it in Part 0. (Measured 2026-07-06.)
r = icd.drop_table(CAT, NS, TBL, purge=True)
print(f"DROP table purgeRequested=true: {r.status_code}")
print("Waiting 10s for the async cleanup task...")
time.sleep(10)

after = mc.list(f"{CAT}/{NS}/{TBL}/")
print(f"Data/metadata files after purge:  {len(after)}")
print()
if len(after) == len(before) and len(before) > 0:
    print("⚠️ VERIFIED: purge removed the catalog record (204) but did NOT delete")
    print("   the data — files orphaned (cleanup task fails, no STS). The DATA persists.")
else:
    print("ℹ️ files changed — if 0, the cleanup task succeeded (check Polaris version).")

Data/metadata files before purge: 14
DROP table purgeRequested=true: 204
Waiting 10s for the async cleanup task...
Data/metadata files after purge:  14

⚠️ VERIFIED: purge removed the catalog record (204) but did NOT delete
   the data — files orphaned (cleanup task fails, no STS). The DATA persists.


## Part 9 — Reclaim storage: delete orphaned data with static MinIO creds

Since the static MinIO credentials CAN delete (no STS needed for that), the
reliable way to reclaim the orphaned data storage is to delete it directly.
Single-object delete (bulk delete needs Content-MD5 on MinIO).


In [17]:
orphaned = mc.list(f"{CAT}/{NS}/{TBL}/", min_size=0)   # include any markers too
print(f"Reclaiming {len(orphaned)} orphaned objects with static MinIO creds...")
deleted = 0
for k in orphaned:
    try:
        mc.delete(k); deleted += 1
    except Exception as e:
        print(f"  could not delete {k}: {str(e)[:60]}")
remaining = mc.list(f"{CAT}/{NS}/{TBL}/")
print(f"Deleted {deleted}; remaining real files: {len(remaining)}")
print()
print("✅ Storage reclaimed via MinIO REST API (single-object DELETEs).")


Reclaiming 14 orphaned objects with static MinIO creds...
Deleted 14; remaining real files: 0

✅ Storage reclaimed via MinIO REST API (single-object DELETEs).


## Part 10 — Summary: the data's journey

In [18]:
print("="*60)
print("DATA LIFECYCLE — what happened to the data")
print("="*60)
print("""
1. WRITE     batch1 → snapshot #1 (5 rows)  Parquet + manifest in MinIO
2. READ      loaded the actual rows back from the snapshot
3. APPEND    batch2 → snapshot #2 — CUMULATIVE: carries forward batch1's
             manifest, so #2 references both files (10 rows total)
4. TIME      read rows AS OF #1 (5 rows) vs #2 (10 rows) — same table,
   TRAVEL    two points in time, real data at each
5. ROLLBACK  moved main ref #2→#1 (back to 5 rows)→#2 (10); pointer only
6. EVOLVE    added column 'category' as a metadata change; data untouched
7. ORPHAN    found unreferenced files by walking snapshot→manifest→data
             (both data files live; superseded manifest-lists are orphans)
             (Polaris API can't do this; it's an Iceberg-reference compare)
8. PURGE     dropped with purge → catalog record gone, but DATA orphaned
             (cleanup task fails without STS — verified)
9. RECLAIM   deleted orphaned data directly with static MinIO creds

KEY DATA CONCEPTS:
  • a snapshot = a complete queryable version (time travel)
  • snapshot → manifest-list → manifests → data files (the reference chain)
  • cumulative append: new snapshot's manifest-list carries forward prior
    manifests, so the table accumulates rather than replaces
  • rollback/evolve change metadata, not the data files
  • orphans = MinIO files no live snapshot references
  • purge does NOT reclaim storage here (no STS); delete orphans yourself
""")


DATA LIFECYCLE — what happened to the data

1. WRITE     batch1 → snapshot #1 (5 rows)  Parquet + manifest in MinIO
2. READ      loaded the actual rows back from the snapshot
3. APPEND    batch2 → snapshot #2 — CUMULATIVE: carries forward batch1's
             manifest, so #2 references both files (10 rows total)
4. TIME      read rows AS OF #1 (5 rows) vs #2 (10 rows) — same table,
   TRAVEL    two points in time, real data at each
5. ROLLBACK  moved main ref #2→#1 (back to 5 rows)→#2 (10); pointer only
6. EVOLVE    added column 'category' as a metadata change; data untouched
7. ORPHAN    found unreferenced files by walking snapshot→manifest→data
             (both data files live; superseded manifest-lists are orphans)
             (Polaris API can't do this; it's an Iceberg-reference compare)
8. PURGE     dropped with purge → catalog record gone, but DATA orphaned
             (cleanup task fails without STS — verified)
9. RECLAIM   deleted orphaned data directly with static MinIO c

In [19]:
# Final cleanup
cat = pc.get_catalog(CAT).json()
if "entityVersion" in cat:
    for ns in ic.list_namespaces(CAT).json().get("namespaces", []):
        nsn = ns[0] if isinstance(ns, list) else ns
        for t in ic.list_tables(CAT, nsn).json().get("identifiers", []):
            ic.drop_table(CAT, nsn, t["name"], purge=True)
        ic.drop_namespace(CAT, nsn)
    pc.delete_catalog(CAT, purge=True)

# The instance principal and its roles are NOT swept by delete_catalog — the
# catalog-role goes with the catalog, but the principal and principal-role are
# instance-scoped and would otherwise leak on every run.
try:
    p, pr, _cr = _inst_names
    pc.delete_principal(p)
    pc.delete_principal_role(pr)
    print(f"   removed instance principal {p} + role {pr}")
except NameError:
    pass

mc.delete_prefix(f"{CAT}/", min_size=0)
print("✅ cleaned up practice catalog + storage")

   removed instance principal data-lifecycle-practice-1632256b-principal + role data-lifecycle-practice-1632256b-prole
✅ cleaned up practice catalog + storage
